# Assignment 2
## [Quantum Computer Programming](https://fagonzalezo.github.io/)

## 1.

The spread of a qubit with state $| \psi \rangle$ with respect to a basis
$\{ | v_0 \rangle, | v_1 \rangle\}$ is defined as $S_{v_0, v_1}(| \psi \rangle) = |\alpha_0| + |\alpha_1|$ where $|\psi \rangle = \alpha_0 |v_0\rangle + \alpha_1 |v_1\rangle$.

Write a function that given a state $| \psi \rangle$ returns a basis $\{ | v_0 \rangle, | v_1 \rangle\}$ such that $S_{v_0, v_1}(| \psi \rangle)$ is maximum.

In [ ]:
import numpy as np
def max_spread(psi):
    '''
     psi: a complex numpy array of shape (2,)
    Returns:
     a list with two complex numpy arrays of shape (2,)
    '''
    # Your code here
    # Ensure the input is a numpy array with complex numbers for precision.
    psi = np.asarray(psi, dtype=np.complex128)

    # For a given state |ψ⟩ = a|0⟩ + b|1⟩, represented as the vector [a, b],
    # a normalized vector orthogonal to it is |ψ_perp⟩ = -b*|0⟩ + a*|1⟩,
    # represented by the vector [-conj(b), conj(a)].
    a, b = psi[0], psi[1]
    psi_perp = np.array([-np.conj(b), np.conj(a)], dtype=np.complex128)

    # The basis that maximizes the spread is formed by the equal superposition
    # of |ψ⟩ and |ψ_perp⟩. This new basis is effectively "rotated" by 45 degrees
    # relative to the basis {|ψ⟩, |ψ_perp⟩}.
    #
    # |v₀⟩ = (|ψ⟩ + |ψ_perp⟩) / √2
    # |v₁⟩ = (|ψ⟩ - |ψ_perp⟩) / √2
    #
    # When |ψ⟩ is expressed in this basis, its coefficients are ⟨v₀|ψ⟩ = 1/√2
    # and ⟨v₁|ψ⟩ = 1/√2. The sum of their magnitudes, |1/√2| + |1/√2|, is √2,
    # which is the maximum possible spread.

    # Pre-calculate 1/√2 for efficiency.
    sqrt2_inv = 1.0 / np.sqrt(2)

    # Construct the two new basis vectors.
    v0 = sqrt2_inv * (psi + psi_perp)
    v1 = sqrt2_inv * (psi - psi_perp)

    return [v0, v1]

## 2.
Given a unitary operator $U$ find a state $|\psi\rangle$ such that $U$ applied to $|\psi\rangle$ produces again $|\psi\rangle$ (ignoring the global phase).

In [ ]:
import numpy as np

def fix_state(U):
    '''
     U: a complex numpy array of shape (2, 2)
    Returns:
     a complex numpy array of shape (2, )
    '''
    # The fixed states under a unitary transformation U are its eigenvectors.
    # The eigenvalues of a unitary matrix have a magnitude of 1, corresponding to a global phase.
    eigenvalues, eigenvectors = np.linalg.eig(U)

    # We can return any of the eigenvectors as they represent the states that are only
    # changed by a global phase when U is applied.
    # For simplicity, let's return the first eigenvector.
    return eigenvectors[:, 0]

## 3.
Suppose a system of $n$ qubits $\{ q_1, \dots, q_n \}$ with state $|\psi\rangle\in\mathbb{C}^{2^n}$.

Write a function $f:\mathbb{C}^{2^n}\times \{1,\dots,n\} \rightarrow \mathbb{R}$ such that $f(|\psi\rangle, i)$ corresponds to the probability that the qubit $q_i$ is in state $|1\rangle$.

In [ ]:
import numpy as np

def prob_f(psi, i):
    '''
     psi: a complex numpy array of shape (2 ** n, )
     i: integer in the range [1, n]
    Returns:
     a real number
    '''
    # Your code here
    # Ensure psi is a numpy array for calculations.
    psi = np.asarray(psi, dtype=np.complex128)
    L = psi.size
    if L == 0 or (L & (L - 1)) != 0:
        raise ValueError("len(psi) must be a power of 2")
    n = int(np.round(np.log2(L)))
    if not (1 <= i <= n):
        raise ValueError(f"i must be in [1, {n}]")
    probs = np.abs(psi)**2
    total = probs.sum()
    if total == 0:
        raise ValueError("psi is the zero vector")
    probs = probs / total

    # q1 is most-significant (leftmost) bit -> bit position = n - i
    shift = n - i
    indices = np.arange(L, dtype=np.int64)
    bit_is_one = ((indices >> shift) & 1) == 1
    return float(probs[bit_is_one].sum())


## 4.
Write a function that given the state of a 3 qubits system determines if it is an entangled state or not

In [ ]:
import numpy as np

def entangled(psi):
    '''
     psi: a complex numpy array of shape (8, )
    Returns:
     boolean indicating whether psi is an entangled state or not
    '''
    norm = np.linalg.norm(psi)
    if norm == 0:
        return False
    psi = psi / norm
    tensor = psi.reshape(2, 2, 2)
    rho_A = np.einsum('ijk, mjk -> im', tensor, np.conj(tensor))
    rho_B = np.einsum('ijk, i l k -> j l', tensor, np.conj(tensor))
    rho_C = np.einsum('ijk, ij m -> k m', tensor, np.conj(tensor))

    def is_pure(rho):
        tr_rho2 = np.trace(np.dot(rho, rho))
        return np.isclose(tr_rho2, 1.0, atol=1e-10)

    is_separable = is_pure(rho_A) and is_pure(rho_B) and is_pure(rho_C)
    return not is_separable